# [TV2] Auto-label trên Colab / Kaggle (GPU)
Chạy lần lượt từng ô từ trên xuống. **Runtime > Change runtime type > T4 GPU** trước khi chạy.

Notebook này chỉ để **sinh nhãn nháp** (detector + SAM 2). Ghép cặp người–xe, copy-paste, tiling chạy được ngay trên máy cá nhân (CPU).

In [ ]:
#@title 1. Tham số (chỉ sửa ô này)
# Cách lấy tool: (a) đã push thư mục tools_tv2 lên GitHub của nhóm -> điền REPO_URL
#                (b) nén tools_tv2 thành tools_tv2.zip, upload lên Drive -> điền TOOLS_ZIP
REPO_URL  = "https://github.com/AnhHoangDepTrai/hackathon_AI20k.git"  #@param {type:"string"}
TOOLS_ZIP = ""   #@param {type:"string"}
# Dữ liệu vào: thư mục trên Drive (ảnh, hoặc gốc dataset có train/valid/test)
DATA_DIR  = "/content/drive/MyDrive/AI20k/Vietnam Traffic Vehicle Detectio.v1i.yolo26"  #@param {type:"string"}
OUT_DIR   = "/content/drive/MyDrive/AI20k/roboflow_v1_11cls"  #@param {type:"string"}
KEEP_LABELS = True        #@param {type:"boolean"}  # True: giữ polygon người vẽ (bộ Roboflow). False: ảnh mới chưa có nhãn
REFINE_HUMAN_MASKS = True #@param {type:"boolean"}  # SAM vẽ lại polygon người vẽ cho sát vật (bộ Roboflow khá thô)
DET_MODEL = "yolo11x.pt"  #@param ["yolo11x.pt", "yolov8x.pt", "yolo11x-seg.pt"]
OPEN_VOCAB = "world"      #@param ["world", "gdino", "none"]
SAM_MODEL = "sam2.1_b.pt" #@param ["sam2.1_t.pt", "sam2.1_b.pt", "sam2.1_l.pt"]
LIMIT = 0                 #@param {type:"integer"}  # >0: chạy thử N ảnh trước

In [ ]:
#@title 2. GPU + Drive
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Lấy tool + cài thư viện
import os, zipfile, subprocess
if TOOLS_ZIP:
    zipfile.ZipFile(TOOLS_ZIP).extractall('/content/tv2')
    TOOLS = next(p for p in ['/content/tv2/tools_tv2', '/content/tv2'] if os.path.exists(p + '/autolabel.py'))
else:
    if not os.path.exists('/content/repo'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
    TOOLS = '/content/repo/tools_tv2'
assert os.path.exists(TOOLS + '/autolabel.py'), 'Không thấy tools_tv2/autolabel.py — kiểm tra REPO_URL/TOOLS_ZIP'
!pip -q install "ultralytics==8.4.214"
if OPEN_VOCAB == "gdino":
    !pip -q install -U transformers
print('TOOLS =', TOOLS)

In [ ]:
#@title 4. Kiểm thử nhanh tool (không cần model, ~5 giây)
!cd "$TOOLS" && python tests/test_tv2.py && python tests/test_pipeline_mock.py | tail -1

In [ ]:
#@title 5. Chạy auto-label (có --resume: Colab ngắt thì chạy lại ô này, nó làm tiếp)
cmd = f'python "{TOOLS}/autolabel.py" --images "{DATA_DIR}" --out "{OUT_DIR}" --det-model {DET_MODEL} --open-vocab {OPEN_VOCAB} --sam-model {SAM_MODEL} --resume'
if KEEP_LABELS: cmd += ' --keep-labels'
if KEEP_LABELS and REFINE_HUMAN_MASKS: cmd += ' --refine-human-masks'
if LIMIT: cmd += f' --limit {LIMIT}'
print(cmd)
!{cmd}

In [ ]:
#@title 6. Xem nhanh vài ảnh kết quả (ảnh nghi ngờ nhất trước)
!python "$TOOLS/visualize.py" --data "$OUT_DIR" --review "$OUT_DIR/review_priority.csv" --n 12 --out /content/viz_check
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/content/viz_check/*.jpg'))[:12]:
    display(Image(f, width=1100))

In [ ]:
#@title 7. Nén kết quả để tải về / đưa lên CVAT để duyệt
import shutil
z = shutil.make_archive(OUT_DIR.rstrip('/') , 'zip', OUT_DIR)
print('Đã nén:', z)